# Experiment 11: Analysis and Implementation of Performance Metrics in Python

**Aim:** To implement and analyze classification performance metrics in Python, verify manual mathematical formulations against scikit-learn on the Cleveland Heart Disease dataset, plot ROC curves, and evaluate clinical metric tradeoffs.

### Dataset Description
**Dataset:** Cleveland Heart Disease Clinical Dataset (297 patient records, 13 diagnostic features, binary diagnosis: 0 = Normal [160], 1 = Heart Disease Present [137]).
- Positive class defined as 1 (Heart Disease Present).

### Theory
Evaluation metrics must align with domain error costs. In binary diagnostic testing, a confusion matrix categorizes outcomes into TP, TN, FP, and FN. Accuracy = (TP+TN)/N measures global correctness but misleads under class imbalance. Precision = TP/(TP+FP) measures positive predictive value (minimizing false alarms). Recall (Sensitivity) = TP/(TP+FN) measures the proportion of actual disease cases identified. Specificity = TN/(TN+FP) measures true negative rate. F1-score is the harmonic mean of precision and recall. ROC curves plot TPR against FPR across all decision thresholds, with AUC quantifying ranking discriminability.

#### 1. Load Dataset, Define Diagnostic Positive Class, and Stratified Split

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_curve, roc_auc_score

cols = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']
df = pd.read_csv('../datasets/cleveland_heart.csv', header=None, names=cols, na_values='?').dropna()
df['target'] = (df['target'] > 0).astype(int)
X, y = df.drop(columns='target'), df['target']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print('Positive Class: Heart Disease (1), Negative Class: Normal (0)')

#### 2. Train Logistic Regression and Decision Tree Models

In [ ]:
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
dt = DecisionTreeClassifier(max_depth=4, random_state=42).fit(Xtr, ytr)
models = {'Logistic Regression': lr, 'Decision Tree': dt}

#### 3. Manual Metric Formulation vs Scikit-Learn Verification

In [ ]:
for name, m in models.items():
    pred = m.predict(Xte)
    proba = m.predict_proba(Xte)[:, 1]
    cm = confusion_matrix(yte, pred)
    tn, fp, fn, tp = cm.ravel()
    # Manual calculation
    acc_m = (tp + tn) / (tp + tn + fp + fn)
    prec_m = tp / (tp + fp)
    rec_m = tp / (tp + fn)
    spec_m = tn / (tn + fp)
    f1_m = 2 * (prec_m * rec_m) / (prec_m + rec_m)
    auc_m = roc_auc_score(yte, proba)
    
    # Assert equality with scikit-learn
    assert np.isclose(acc_m, accuracy_score(yte, pred))
    assert np.isclose(prec_m, precision_score(yte, pred))
    assert np.isclose(rec_m, recall_score(yte, pred))
    assert np.isclose(f1_m, f1_score(yte, pred))
    
    print(f'=== {name} ===')
    print(f'TP={tp}, FN={fn}, FP={fp}, TN={tn}')
    print(f'Accuracy:    {acc_m:.4f} (Verified)')
    print(f'Precision:   {prec_m:.4f} (Verified)')
    print(f'Recall:      {rec_m:.4f} (Sensitivity)')
    print(f'Specificity: {spec_m:.4f}')
    print(f'F1-Score:    {f1_m:.4f}')
    print(f'ROC-AUC:     {auc_m:.4f}\n')

#### 4. Comparative ROC Curves Visualization

In [ ]:
plt.figure(figsize=(7, 6))
for name, m in models.items():
    proba = m.predict_proba(Xte)[:, 1]
    fpr, tpr, _ = roc_curve(yte, proba)
    auc = roc_auc_score(yte, proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Chance Line (AUC = 0.5000)')
plt.title('Receiver Operating Characteristic (ROC) Comparison')
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Recall / Sensitivity)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Observations
1. Manual arithmetic calculations for Accuracy, Precision, Recall, and F1 strictly matched scikit-learn outputs to machine precision.
2. Logistic Regression demonstrated superior performance across all metrics:
   - Accuracy: 83.33% vs 75.00% (Decision Tree)
   - Recall (Sensitivity): 78.57% (22/28 disease cases detected) vs 67.86% (19/28)
   - Specificity: 87.50% (28/32 normal cases identified) vs 81.25% (26/32)
   - ROC-AUC: 0.9498 vs 0.7919
3. In clinical cardiac screening, Recall is the paramount metric because a False Negative (missed coronary heart disease) poses immediate mortal risk, whereas a False Positive merely prompts a non-invasive confirmatory angiogram.

### Result
Logistic Regression significantly outperformed the Decision Tree on this diagnostic dataset, achieving an exceptional ROC-AUC of 0.9498 and Sensitivity of 78.57%. Performance metric selection must prioritize clinical consequences over unweighted global accuracy.